In [1]:
import pandas as pd
import geopandas as gpd

# Definiamo prima di tutto il percorso relativo del nostro file GeoJSON
# Siccome notebook si trova nella cartella /notebooks, utilizziamo '../'
# per risalire alla directory principale ed entrare all'interno della cartella /data

percorso_file = "../data/processed/dataset_parchi_salerno.geojson"


# Ora carichiamo i dati nel GeoDataFrame
print("Caricamento dei dati in corso...")
gdf_parchi = gpd.read_file(percorso_file)

# Verifichiamo ora l'integrità dei dati
print("I dati sono stati caricati con successo")
print(f"Numero totale di lotti analizzati: {len(gdf_parchi)}")


# Mostriamo le prime 3 righe per verificare che la tabella attributi sia presente
gdf_parchi.head(3)


Caricamento dei dati in corso...
I dati sono stati caricati con successo
Numero totale di lotti analizzati: 317


,country,fua_name,fua_code,code_2018,class_2018,prod_date,identifier,perimeter,area,comment,DN,Area_mq,Centroid_X,Centroid_Y,Area_Clean,Perim_m,geometry
0,IT,Salerno,IT032L3,12100,"Industrial, commercial, public, military and p...",2025-09,5829-IT032L3,567.411740,10369.221876,None,1,10369.220,487482.970,4499498.244,10369.220,567.404,"POLYGON ((487449.325 4499488.723, 487449.319 4..."
1,IT,Salerno,IT032L3,12100,"Industrial, commercial, public, military and p...",2025-09,5841-IT032L3,457.327045,11248.447610,None,1,11248.445,487549.779,4499330.589,11248.445,457.487,"POLYGON ((487556.529 4499270.187, 487548.459 4..."
2,IT,Salerno,IT032L3,12100,"Industrial, commercial, public, military and p...",2025-09,5820-IT032L3,172.640718,1681.106842,None,1,1611.227,487291.427,4499257.476,1611.227,187.623,"POLYGON ((487287.298 4499232.163, 487269.326 4..."


In [2]:
import numpy as np


# Calcoliamo l'indice di Compattezza (Polsby-Popper)

In [3]:
import numpy as np


# Calcoliamo l'indice di Compattezza (Polsby-Popper)
# Questo indice restituisce un valore tra 0 e 1 per poter valutare
# la forma dell'area che stiamo andando a considerare

gdf_parchi['Compattezza'] = (4 * np.pi * gdf_parchi['Area_Clean']) / (gdf_parchi['Perim_m'] ** 2)

# Ottenuto l'indice di compattezza per ognuna delle zone considerate
# Scartiamo ora i vari corridoi stradali tenendo conto solo delle
# aree con compattezza maggiore o uguale di 0.15

soglia_compattezza = 0.15
aree_scartate = len(gdf_parchi[gdf_parchi['Compattezza'] < soglia_compattezza])

gdf_parchi_validi = gdf_parchi[gdf_parchi['Compattezza'] >= soglia_compattezza].copy()

# Ora possiamo effettuare la categorizzazione urbanistica (Binning)
# Definiamo i vari limiti: 
# da 1000 a 10000 mq rientriamo nella categoria "Parco di Quartiere"
# da 10000 mq in su rientriamo nella categoria "Grande Parco Urbano"

limiti = [1000, 10000, float('inf')]
etichette = ['Parco di Quartiere','Grande Parco Urbano']

gdf_parchi_validi['Categoria'] = pd.cut(gdf_parchi_validi['Area_Clean'], bins=limiti, labels=etichette, right=False)

# Report finale successivo all'utilizzo dei vari indici e delle categorie
print("REPORT FEATURE ENGINEERING:")
print(f"Ecco elencate le aree scartate (corridoi/strade anomale): {aree_scartate}")
print(f"Invece di seguito il numero delle aree rimanenti considerate valide per il modello IA: {len(gdf_parchi_validi)}\n")

print("Distribuzione delle due Categorie Urbanistiche individuate:")
print(gdf_parchi_validi['Categoria'].value_counts())


# Anteprima delle nuove colonne create per tener traccia della selezione effettuata

gdf_parchi_validi[['identifier','Area_Clean','Perim_m','Compattezza','Categoria']].head()

REPORT FEATURE ENGINEERING:
Ecco elencate le aree scartate (corridoi/strade anomale): 21
Invece di seguito il numero delle aree rimanenti considerate valide per il modello IA: 296

Distribuzione delle due Categorie Urbanistiche individuate:
Categoria
Parco di Quartiere     196
Grande Parco Urbano    100
Name: count, dtype: int64


,identifier,Area_Clean,Perim_m,Compattezza,Categoria
0,5829-IT032L3,10369.220,567.404,0.404735,Grande Parco Urbano
1,5841-IT032L3,11248.445,457.487,0.675375,Grande Parco Urbano
2,5820-IT032L3,1611.227,187.623,0.575168,Parco di Quartiere
3,5811-IT032L3,15114.577,734.684,0.351888,Grande Parco Urbano
4,5821-IT032L3,3284.036,404.720,0.251947,Parco di Quartiere


In [4]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

# Normalizziamo i dati
# Prendiamo i valori dell'area e della compattezza e li andiamoa rendere confrontabili in
# una scala che va da 0 a 1 così da essere confrontabili
# Per fare ciò inizializziamo lo scaler per portare i valori
# in un range che va da 0 ad 1 : [0, 1]

scaler = MinMaxScaler()
gdf_parchi_validi[['Area_Norm', 'Comp_Norm']] = scaler.fit_transform(gdf_parchi_validi[['Area_Clean','Compattezza']])

# Scoring Multi-Criterio (MCDA)
# Calcoliamo il "Suitability Score" ovvero il punteggio di Idoneità da 0 a 100
# Traducendo così la logica urbana in numeri, dando leggermente più importanza all'area che alla forma del lotto
# I Pesi Logici: 60% estensione (Area) e 40% forma (Compattezza)
peso_area = 0.6
peso_compattezza = 0.4

gdf_parchi_validi['Suitability_Score'] = (gdf_parchi_validi['Area_Norm'] * peso_area + gdf_parchi_validi['Comp_Norm'] * peso_compattezza) *100


# K-Means Clustering
# Usiamo le feature normalizzate per l'algoritmo di raggruppamento
# Rimuoviamo i bias umani lasciando ad analizzare all'algoritmo i lotti in tre fasce
# Permettiamo così di creare Tre Cluster tra i lotti disponibili in base a tre Fasce di priorità

features_per_ai = gdf_parchi_validi[['Suitability_Score','Area_Norm','Comp_Norm']]

# Utilizziamo random_state con valore 42 per garantire la riproducibilità esatta dei risultati

kmeans = KMeans(n_clusters = 3, random_state = 42, n_init = 10)
gdf_parchi_validi['Cluster_ID'] = kmeans.fit_predict(features_per_ai)

# Ora avremo i cluster numerati in ordine casuale (0, 1, 2)
# Verranno quindi rinominati trovando quale cluster ha la media di Score più alta

media_score_cluster = gdf_parchi_validi.groupby('Cluster_ID')['Suitability_Score'].mean().sort_values()

# Quindi mappiamo i cluster rendendo leggibili per i decisori i risultati
mappa_nomi = {
    media_score_cluster.index[0]: 'Priorità Bassa',
    media_score_cluster.index[1]: 'Priorità Media',
    media_score_cluster.index[2]: 'Priorità Alta'
}


gdf_parchi_validi['Fascia_Idoneità'] = gdf_parchi_validi['Cluster_ID'].map(mappa_nomi)

# Filtriamo ora solo la fascia Alta e ordiniamo per la il punteggio migliore i risultati ottenuti
top_10 = gdf_parchi_validi[gdf_parchi_validi['Fascia_Idoneità'] == 'Priorità Alta'].sort_values(by='Suitability_Score', ascending = False).head(10)

# Ora mostriamo il risultato finale formattato come Tabella

print("Classifica della top 10 aree migliori per Salerno\n")
display(top_10[['identifier','Categoria', 'Area_Clean','Compattezza','Suitability_Score','Fascia_Idoneità']])



Classifica della top 10 aree migliori per Salerno



,identifier,Categoria,Area_Clean,Compattezza,Suitability_Score,Fascia_Idoneità
11,5854-IT032L3,Grande Parco Urbano,218152.148,0.173122,61.126820,Priorità Alta
143,5874-IT032L3,Grande Parco Urbano,173058.758,0.336168,58.270420,Priorità Alta
33,5926-IT032L3,Grande Parco Urbano,201935.047,0.170407,56.485983,Priorità Alta
196,5692-IT032L3,Grande Parco Urbano,112195.805,0.534620,53.142163,Priorità Alta
113,5694-IT032L3,Grande Parco Urbano,55304.052,0.794002,52.699933,Priorità Alta
298,5796-IT032L3,Grande Parco Urbano,65467.396,0.722615,51.303499,Priorità Alta
147,5869-IT032L3,Grande Parco Urbano,95472.777,0.560247,50.030877,Priorità Alta
60,5824-IT032L3,Grande Parco Urbano,70089.179,0.655289,48.615128,Priorità Alta
13,5834-IT032L3,Grande Parco Urbano,160390.283,0.219363,47.890348,Priorità Alta
158,5867-IT032L3,Grande Parco Urbano,61726.024,0.676606,47.559869,Priorità Alta


In [6]:
import folium

# Preparazione dei dati per il WebGIS
# In questa fase Riproiettiamo i dati da coordinate piane (UTM 33N) a coordinate 
# geografiche (WGS84 Lat/Lon) per il Web

gdf_wgs84 = gdf_parchi_validi.to_crs(epsg=4326)
top_10_wgs84 = top_10.to_crs(epsg=4326)

# In questa sezione creiamo la mappa base centrata su Salerno
# Calcolando il Centroide Medio di tutte le aree per centrare la vista perfettamente su di esse
centro_geografico = gdf_parchi_validi.geometry.centroid.to_crs(epsg=4326)
centro_y = centro_geografico.y.mean()
centro_x = centro_geografico.x.mean()

mappa_salerno = folium.Map(location=[centro_y, centro_x], zoom_start=13, tiles='CartoDB positron')

# Aggiungiamo ora tutte le aree valide in grigio

folium.GeoJson(
    gdf_wgs84,
    style_function= lambda x:{'fillColor': '#cccccc', 'color': '#999999', 'weight': 1, 'fillOpacity' : 0.4},
    name = "Tutte le Aree Candidate"
).add_to(mappa_salerno)


# Aggiungiamo la Top 10 in Verde con i Pop-Up Interattivi
folium.GeoJson(
    top_10_wgs84,
    style_function=lambda x: {'fillColor': '#28a745', 'color': '#1e7e34', 'weight':2,'fillOpacity':0.8},
    tooltip = folium.GeoJsonTooltip(
        fields=['identifier','Categoria', 'Suitability_Score'],
        aliases=['ID Lotto:', 'Categoria:', 'Punteggio:'],
        localize=True
    ),
    name = "Top 10 Aree"
).add_to(mappa_salerno)


# Inseriamo un controllo per poter accendere/spegnere i layer
folium.LayerControl().add_to(mappa_salerno)

# Effettuiamo in questo punto il salvataggio della mappa HTML
percorso_mappa = "../mappa_interattiva_salerno.html"
mappa_salerno.save(percorso_mappa)
print(f"Mappa WebGIS salvata correttamente in :{percorso_mappa}")

# Salvataggio finale del GeoJSON con tutti i calcoli effettuati fino ad ora

percorso_output = "../data/processed/risultati_finali_salerno.geojson"
gdf_parchi_validi.to_file(percorso_output, driver="GeoJSON")
print(f"Dataset definitivo (con score IA) salvato in {percorso_output}\n")

# Mostriamo la mappa direttamente qui nel notebook
mappa_salerno

Mappa WebGIS salvata correttamente in :../mappa_interattiva_salerno.html
Dataset definitivo (con score IA) salvato in ../data/processed/risultati_finali_salerno.geojson

